# 12 - Phase 4 Method Comparison

Single-duration comparison notebook for a 2h BESS under medium Phase 4 settings.

Methods included:
- **Initial hourly intrinsic**: HPFC daily LP from notebook 11, renamed from HPFC perfect-foresight dispatch.
- **DA rolling intrinsic**: HPFC-anchored simulated DA paths, 48 half-hour look-ahead and 48 half-hour roll step.
- **WD rolling intrinsic**: HPFC-anchored simulated DA paths, 48 half-hour look-ahead and 8 half-hour roll step.
- **Forward simulation**: LSMC backward policy plus out-of-sample forward application on HPFC-anchored simulated DA paths.
- **Perfect foresight**: full-horizon perfect-foresight LP on sampled HPFC-anchored simulated DA paths.

Configured here as:
```python
PHASE4_RUN_MODE_FOR_SWEEP = "medium"
SWEEP_DURATIONS_H = [2.0]
```

The simulated methods use the HPFC hourly curve from the initial hourly intrinsic method as the deterministic anchor, then apply relative stochastic movements from the Phase 3 simulation bundle.


In [8]:
from __future__ import annotations

import copy
import json
import os
import pickle
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.config import ASSET, DEGRADATION, FINANCE, LSMC as LSMC_CFG, SCHWARTZ_SMITH, configure_asset_duration
from src.optimisation.dispatch import enumerate_modes
from src.optimisation.lsmc import LSMCSolver
from src.optimisation.perfect_foresight import solve_perfect_foresight
from src.optimisation.rolling_intrinsic import rolling_intrinsic_parallel, rolling_intrinsic, solve_daily_lp
from src.processes.ancillary import AncillaryParams
from src.processes.hpfc import HPFCParams
from src.processes.imbalance import ImbalanceParams
from src.processes.schwartz_smith import SSParams
from src.processes.simulate import PathBundle, default_params_from_config
from src.validation import summarize_action_distribution, validate_path_bundle

RAW = PROJECT_ROOT / 'data' / 'raw'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

PHASE4_RUN_MODE_FOR_SWEEP = 'medium'
SWEEP_DURATIONS_H = [2.0]
VALUATION_DURATION_H = SWEEP_DURATIONS_H[0]
SEED = int(LSMC_CFG.get('seed', 42))

# Keep the notebook compatible with the duration-sweep convention.
config.VALID_ASSET_DURATIONS_H = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0)
ASSET_VAL = copy.deepcopy(ASSET)
configure_asset_duration(ASSET_VAL, VALUATION_DURATION_H)
DURATION_LABEL = f'{VALUATION_DURATION_H:g}h'

PHASE4_RUNS = {
    'medium': {
        'ri_paths': 50,
        'bwd_paths': 500,
        'bwd_steps': 4320,
        'fwd_paths': 500,
        'fwd_workers': 6,
        'ridge_alpha': 200.0,
        'continuation_value_cap_gbp': 3_000_000,
        'n_soc_nodes': 9,
        'soh_nodes': [1.00, 0.90, 0.82],
        'net_levels': [-1.0, -0.5, 0.0, 0.5, 1.0],
        'dc_levels': [0.0, 0.5],
        'qr_levels': [0.0, 0.25],
    }
}
PHASE4_RUN = dict(PHASE4_RUNS[PHASE4_RUN_MODE_FOR_SWEEP])

print(f'Project root: {PROJECT_ROOT}')
print(f'Run mode: {PHASE4_RUN_MODE_FOR_SWEEP}')
print(f'Duration: {DURATION_LABEL} ({ASSET_VAL["power_mw"]:.0f} MW / {ASSET_VAL["energy_mwh"]:.0f} MWh)')
print(f'Seed: {SEED}')



Project root: g:\My Drive\Research\bess_project
Run mode: medium
Duration: 2h (100 MW / 200 MWh)
Seed: 42


## 1  Load Simulation Bundle and Parameters


In [9]:
def _load_json_or_default(cls, default_obj, *fnames):
    for fname in fnames:
        path = PROCESSED / fname
        if path.exists():
            print(f'Loaded calibrated {fname}')
            return cls.from_json(path)
    print(f'Using config default for {" or ".join(fnames)}')
    return default_obj

ss_def, hpfc_def, imb_def, anc_def = default_params_from_config()
ss_p = _load_json_or_default(SSParams, ss_def, 'ss_params.json')
hpfc_p = _load_json_or_default(HPFCParams, hpfc_def, 'pca_params.json', 'hpfc_params.json')
imb_p = _load_json_or_default(ImbalanceParams, imb_def, 'imbalance_params.json')
anc_p = _load_json_or_default(AncillaryParams, anc_def, 'ancillary_params.json')

SIM_BUNDLE_PATH = PROCESSED / 'sim_bundle.pkl'
if not SIM_BUNDLE_PATH.exists():
    raise FileNotFoundError(f'Missing simulation bundle: {SIM_BUNDLE_PATH}. Run notebooks/03_simulation.ipynb first.')

with SIM_BUNDLE_PATH.open('rb') as f:
    bundle = pickle.load(f)

bundle_check = validate_path_bundle(bundle, forward_anchor_gbp_mwh=SCHWARTZ_SMITH['forward_anchor_gbp_mwh'])
bundle_check.raise_if_failed()
print(bundle_check.summary())

RUN_STEPS = min(int(PHASE4_RUN['bwd_steps'] or bundle.n_steps), bundle.n_steps)
RUN_BWD_PATHS = min(int(PHASE4_RUN['bwd_paths'] or bundle.n_paths), bundle.n_paths)
RUN_FWD_PATHS = min(int(PHASE4_RUN['fwd_paths'] or bundle.n_paths), bundle.n_paths)
RI_N_PATHS = min(int(PHASE4_RUN['ri_paths']), bundle.n_paths)
DT_H = float(LSMC_CFG.get('dt_hours', 0.5))
valuation_horizon_years = RUN_STEPS * DT_H / 8760.0
annualization_factor = 1.0 / valuation_horizon_years

ln_P_bundle_mat = np.asarray(bundle.ln_P_base[:, :RUN_STEPS + 1], dtype=np.float32)
P_da_bundle_mat = np.exp(np.clip(ln_P_bundle_mat, -100.0, np.log(500.0))).astype(np.float32)
rng = np.random.default_rng(SEED)
ri_idx = rng.choice(bundle.n_paths, size=RI_N_PATHS, replace=False)

# Filled after the HPFC curve is loaded in the initial hourly intrinsic section.
P_da_anchored_mat = None
ln_P_anchored_mat = None
P_da_ri = None

print(f'Selected horizon: {RUN_STEPS:,} half-hours = {valuation_horizon_years:.3f} years')
print(f'RI/PF sample: {RI_N_PATHS:,} paths')
print(f'LSMC sample: backward {RUN_BWD_PATHS:,}, forward {RUN_FWD_PATHS:,} disjoint paths')


Loaded calibrated ss_params.json
Loaded calibrated pca_params.json
Loaded calibrated imbalance_params.json
Loaded calibrated ancillary_params.json
path_bundle: PASS
Selected horizon: 4,320 half-hours = 0.247 years
RI/PF sample: 50 paths
LSMC sample: backward 500, forward 500 disjoint paths


## 2  Initial Hourly Intrinsic (HPFC Daily LP)


In [ ]:
# This method is the HPFC daily LP from notebook 11, renamed here as initial hourly intrinsic.
# It values deterministic hourly HPFC arbitrage over a 5-year forward-curve horizon.
DATA_PROC = PROCESSED
DOW_NAMES = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
HORIZON_YEARS = 5
APPLY_MARKET_DECAY = True
# Fixed as-of date — keeps the HPFC filter and decay-year offsets reproducible
# regardless of when the notebook is re-run.
AS_OF_DATE = pd.Timestamp('2026-05-03')
HORIZON_END = AS_OF_DATE + pd.DateOffset(years=HORIZON_YEARS)

def calendar_weighted_mean(year: int, month: int, mult: pd.DataFrame) -> float:
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthEnd(0)
    total, count = 0.0, 0
    for d in pd.date_range(start, end, freq='D'):
        total += float(mult[DOW_NAMES[d.dayofweek]].sum())
        count += 24
    return total / count

def load_or_build_hpfc_full() -> tuple[pd.DataFrame, pd.Timestamp]:
    hpfc_path = DATA_PROC / 'hpfc.parquet'
    shape_path = DATA_PROC / 'shape_multipliers.parquet'
    if not hpfc_path.exists() or not shape_path.exists():
        raise FileNotFoundError('Missing hpfc.parquet or shape_multipliers.parquet. Run notebooks/11_hourly_forward_curve.ipynb first.')
    hpfc = pd.read_parquet(hpfc_path)
    hpfc['delivery_date'] = pd.to_datetime(hpfc['delivery_date'])
    hpfc_fut = hpfc[hpfc['delivery_date'] >= AS_OF_DATE].copy()
    hpfc_cov = hpfc_fut['delivery_date'].max()
    shape = pd.read_parquet(shape_path)
    raw_mult = shape.pivot(index='hour', columns='dow_name', values='multiplier')[DOW_NAMES]
    gap_days = max(0, (HORIZON_END - hpfc_cov).days - 1)
    if gap_days <= 0:
        return hpfc_fut[hpfc_fut['delivery_date'] <= HORIZON_END].copy(), hpfc_cov

    trailing_start = hpfc_cov - pd.DateOffset(months=12)
    flat_px = float(hpfc_fut[hpfc_fut['delivery_date'] >= trailing_start]['monthly_fwd_gbp_mwh'].mean())
    ext_rows = []
    for d in pd.date_range(hpfc_cov + pd.Timedelta(days=1), HORIZON_END, freq='D'):
        cm = calendar_weighted_mean(d.year, d.month, raw_mult)
        col = DOW_NAMES[d.dayofweek]
        for h in range(24):
            m = float(raw_mult.loc[h, col]) / cm
            ext_rows.append({
                'delivery_date': d,
                'hour': h,
                'dow': d.dayofweek,
                'delivery_month': d.strftime('%Y-%m'),
                'monthly_fwd_gbp_mwh': flat_px,
                'multiplier': m,
                'price_gbp_mwh': flat_px * m,
            })
    return pd.concat([hpfc_fut, pd.DataFrame(ext_rows)], ignore_index=True), hpfc_cov

def run_initial_hourly_intrinsic(asset_cfg: dict) -> dict:
    hpfc_full, hpfc_cov = load_or_build_hpfc_full()
    hpfc_full = hpfc_full.sort_values(['delivery_date', 'hour']).reset_index(drop=True)
    decay = float(FINANCE['revenue_decay_per_year'])
    p_bar = float(asset_cfg['power_mw'])
    e_nm = float(asset_cfg['energy_mwh'])
    eta_c = float(asset_cfg['eta_charge'])
    eta_d = float(asset_cfg['eta_discharge'])
    e_min = float(asset_cfg['soc_min_mwh'])
    e_max = float(asset_cfg['soc_max_mwh'])
    avail = float(asset_cfg['availability'])
    e_t = float(asset_cfg['soc_init_mwh'])
    daily_rows = []
    for d, grp in hpfc_full.groupby('delivery_date'):
        prices = grp.sort_values('hour')['price_gbp_mwh'].to_numpy()
        if len(prices) < 24:
            continue
        d_opt, c_opt, gross_rev = solve_daily_lp(prices, e_t, e_min, e_max, p_bar, eta_c, eta_d, dt_h=1.0)
        e_t = float(np.clip(e_t + (c_opt * eta_c - d_opt / eta_d).sum(), e_min, e_max))
        daily_rows.append({
            'date': pd.Timestamp(d),
            'gross_revenue_gbp': gross_rev * avail,
            'cycles_equiv': float(d_opt.sum() / e_nm) * avail,
        })
    daily = pd.DataFrame(daily_rows)
    daily['years_elapsed'] = (daily['date'] - AS_OF_DATE).dt.days / 365.25
    daily['decay_factor'] = (1 - decay) ** daily['years_elapsed'] if APPLY_MARKET_DECAY else 1.0
    daily['gross_adj_gbp'] = daily['gross_revenue_gbp'] * daily['decay_factor']
    annual = daily.groupby(((daily['date'] - AS_OF_DATE).dt.days // 365.25).astype(int)).agg(
        gross_gbp=('gross_adj_gbp', 'sum'), cycles=('cycles_equiv', 'sum')
    ).head(HORIZON_YEARS)
    annual_mean = float(annual['gross_gbp'].mean())
    return {
        'method': 'Initial hourly intrinsic',
        'value_gbp_horizon_mean': annual_mean,
        'value_gbp_annualized_mean': annual_mean,
        'gbp_per_mw_year': annual_mean / asset_cfg['power_mw'],
        'n_paths': 1,
        'window_hh': 24,
        'gate_hh': 24,
        'pv_paths': np.array([annual_mean]),
        'notes': f'HPFC hourly daily LP; 5-year annual avg gross revenue as-of {AS_OF_DATE.date()}; energy-only',
    }

initial_hourly_row = run_initial_hourly_intrinsic(ASSET_VAL)
initial_hourly_row


def build_hpfc_half_hour_anchor(hpfc_full: pd.DataFrame) -> np.ndarray:
    repeats_per_hour = int(round(1.0 / DT_H))
    if not np.isclose(repeats_per_hour * DT_H, 1.0):
        raise ValueError(f'DT_H={DT_H} does not divide one hour cleanly.')
    hourly = (
        hpfc_full.sort_values(['delivery_date', 'hour'])['price_gbp_mwh']
        .to_numpy(dtype=float)
    )
    hh = np.repeat(hourly, repeats_per_hour)
    if len(hh) < RUN_STEPS + 1:
        hh = np.pad(hh, (0, RUN_STEPS + 1 - len(hh)), mode='edge')
    return np.clip(hh[:RUN_STEPS + 1], 1e-3, 500.0).astype(np.float32)


def build_hpfc_anchored_simulation_prices(anchor_curve: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # Use the simulation bundle for relative shocks, but reset each path to start
    # from the HPFC curve level. This keeps the intrinsic and simulated methods
    # in the same price universe.
    rel_log = ln_P_bundle_mat - ln_P_bundle_mat[:, :1]
    relative_move = np.exp(np.clip(rel_log, -3.0, 3.0))
    prices = np.clip(anchor_curve[None, :] * relative_move, 1e-3, 500.0).astype(np.float32)
    return prices, np.log(prices).astype(np.float32)


hpfc_full_for_anchor, hpfc_anchor_coverage = load_or_build_hpfc_full()
hpfc_half_hour_anchor = build_hpfc_half_hour_anchor(hpfc_full_for_anchor)
P_da_anchored_mat, ln_P_anchored_mat = build_hpfc_anchored_simulation_prices(hpfc_half_hour_anchor)
P_da_ri = P_da_anchored_mat[ri_idx, :RUN_STEPS].astype(np.float32)

print(f'HPFC anchor: {len(hpfc_half_hour_anchor):,} half-hours; first price GBP{hpfc_half_hour_anchor[0]:.2f}/MWh')
print(f'Anchored DA paths: mean GBP{P_da_anchored_mat[:, :RUN_STEPS].mean():.1f}/MWh, std GBP{P_da_anchored_mat[:, :RUN_STEPS].std():.1f}/MWh')


## 3  Rolling Intrinsic Benchmarks


In [ ]:
def summarise_path_values(method: str, values: np.ndarray, n_paths: int, window_hh: int, gate_hh: int, notes: str) -> dict:
    mean_horizon = float(np.mean(values))
    ann = mean_horizon * annualization_factor
    return {
        'method': method,
        'value_gbp_horizon_mean': mean_horizon,
        'value_gbp_annualized_mean': ann,
        'gbp_per_mw_year': ann / ASSET_VAL['power_mw'],
        'n_paths': int(n_paths),
        'window_hh': int(window_hh),
        'gate_hh': int(gate_hh),
        'pv_paths': np.asarray(values) * annualization_factor,   # annualised per-path values
        'notes': notes,
    }

def run_rolling_intrinsic_case(name: str, window_hh: int, gate_hh: int) -> tuple[dict, np.ndarray]:
    print(f'Running {name}: window_hh={window_hh}, gate_hh={gate_hh}, paths={RI_N_PATHS}')
    workers = max(1, min(8, (os.cpu_count() or 2) - 1, RI_N_PATHS))
    t0 = time.time()
    try:
        pv, soc = rolling_intrinsic_parallel(
            P_da_ri,
            ASSET_VAL,
            LSMC_CFG,
            FINANCE,
            E_init_frac=0.5,
            window_hh=window_hh,
            gate_hh=gate_hh,
            max_workers=workers,
            backend='thread',
            verbose=True,
        )
    except Exception as exc:
        print(f'Parallel rolling intrinsic failed ({exc}); falling back to serial')
        pv, soc = rolling_intrinsic(
            P_da_ri,
            ASSET_VAL,
            LSMC_CFG,
            FINANCE,
            E_init_frac=0.5,
            window_hh=window_hh,
            gate_hh=gate_hh,
            verbose=True,
        )
    print(f'{name} complete in {time.time() - t0:.1f}s; mean horizon GBP{np.mean(pv):,.0f}')
    return summarise_path_values(
        name,
        pv,
        RI_N_PATHS,
        window_hh,
        gate_hh,
        f'HPFC-anchored simulated DA paths ({RI_N_PATHS} paths); rolling deterministic LP; energy-only, VOM=0',
    ), pv

da_rolling_row, da_rolling_pv = run_rolling_intrinsic_case('DA rolling intrinsic', window_hh=48, gate_hh=48)
wd_rolling_row, wd_rolling_pv = run_rolling_intrinsic_case('WD rolling intrinsic', window_hh=48, gate_hh=8)
[da_rolling_row, wd_rolling_row]


## 4  Forward Simulation (LSMC Policy)


In [ ]:
LSMC_FAST_CFG = dict(LSMC_CFG)
LSMC_FAST_CFG.update({
    'n_soc_nodes': int(PHASE4_RUN['n_soc_nodes']),
    'soh_nodes': PHASE4_RUN['soh_nodes'],
    'n_soh_nodes': len(PHASE4_RUN['soh_nodes']),
    'ridge_alpha': float(PHASE4_RUN['ridge_alpha']),
    'continuation_value_cap_gbp': float(PHASE4_RUN['continuation_value_cap_gbp']),
})
FAST_MODES = enumerate_modes(
    net_levels=PHASE4_RUN['net_levels'],
    dc_levels=PHASE4_RUN['dc_levels'],
    qr_levels=PHASE4_RUN['qr_levels'],
)

rng_lsmc = np.random.default_rng(SEED)
bwd_idx = rng_lsmc.choice(bundle.n_paths, size=RUN_BWD_PATHS, replace=False)
bundle_bwd = PathBundle(
    chi=bundle.chi[bwd_idx, :RUN_STEPS + 1],
    xi=bundle.xi[bwd_idx, :RUN_STEPS + 1],
    ln_P_base=ln_P_anchored_mat[bwd_idx, :RUN_STEPS + 1],
    lam=bundle.lam[bwd_idx, :RUN_STEPS + 1, :],
    delta_imb=bundle.delta_imb[bwd_idx, :RUN_STEPS + 1],
    pi={k: v[bwd_idx, :RUN_STEPS + 1] for k, v in bundle.pi.items()},
    dt=bundle.dt,
    n_paths=RUN_BWD_PATHS,
    n_steps=RUN_STEPS,
)

solver = LSMCSolver(ASSET_VAL, LSMC_FAST_CFG, DEGRADATION, FINANCE, modes=FAST_MODES, verbose=True, hpfc_params=hpfc_p)
print(f'Backward: {RUN_BWD_PATHS:,} paths x {RUN_STEPS:,} steps; modes={len(FAST_MODES)}')
t0 = time.time()
policy = solver.backward(bundle_bwd)
print(f'Backward complete in {time.time() - t0:.1f}s')
print(f'cont_beta shape: {policy.cont_beta.shape if policy.cont_beta is not None else "None (legacy)"}')

remaining_idx = np.setdiff1d(np.arange(bundle.n_paths), bwd_idx)
fwd_n_paths = min(RUN_FWD_PATHS, len(remaining_idx))
fwd_idx = np.random.default_rng(SEED + 1).choice(remaining_idx, size=fwd_n_paths, replace=False)
bundle_fwd = PathBundle(
    chi=bundle.chi[fwd_idx, :RUN_STEPS + 1],
    xi=bundle.xi[fwd_idx, :RUN_STEPS + 1],
    ln_P_base=ln_P_anchored_mat[fwd_idx, :RUN_STEPS + 1],
    lam=bundle.lam[fwd_idx, :RUN_STEPS + 1, :],
    delta_imb=bundle.delta_imb[fwd_idx, :RUN_STEPS + 1],
    pi={k: v[fwd_idx, :RUN_STEPS + 1] for k, v in bundle.pi.items()},
    dt=bundle.dt,
    n_paths=fwd_n_paths,
    n_steps=RUN_STEPS,
)

workers = max(1, min(int(PHASE4_RUN['fwd_workers']), (os.cpu_count() or 2) - 1, fwd_n_paths))
print(f'Forward simulation: {fwd_n_paths:,} disjoint paths x {RUN_STEPS:,} steps; workers={workers}')
t0 = time.time()
try:
    lsmc_result = solver.forward_parallel(bundle_fwd, policy, max_workers=workers)
except Exception as exc:
    print(f'Parallel forward failed ({exc}); falling back to serial')
    lsmc_result = solver.forward(bundle_fwd, policy)
print(f'Forward complete in {time.time() - t0:.1f}s')

lsmc_ann_pv = lsmc_result.pv_paths * annualization_factor
forward_row = {
    'method': 'Forward simulation (LSMC)',
    'value_gbp_horizon_mean': float(np.mean(lsmc_result.pv_paths)),
    'value_gbp_annualized_mean': float(np.mean(lsmc_ann_pv)),
    'gbp_per_mw_year': float(np.mean(lsmc_ann_pv)) / ASSET_VAL['power_mw'],
    'n_paths': fwd_n_paths,
    'window_hh': 0,
    'gate_hh': 0,
    'pv_paths': lsmc_ann_pv,
    'notes': (
        f'Non-anticipative LSMC policy on HPFC-anchored disjoint paths ({fwd_n_paths} paths); '
        f'full-stack: DA+ancillary, VOM={ASSET_VAL.get("vom_gbp_mwh", 1.2):.2f} £/MWh, degradation cost'
    ),
}
forward_row


## 5  Perfect Foresight (Full-Horizon Simulated DA Paths)


In [ ]:
PF_N_PATHS = RI_N_PATHS
# Perfect-foresight is an energy-only DA upper benchmark: VOM=0, terminal SoC
# pinned to initial SoC so the optimizer can't harvest stranded inventory value
# that the other methods also lose at horizon end.
PF_TERMINAL_SOC = float(ASSET_VAL['soc_init_mwh'])

print(f'Running perfect foresight on {PF_N_PATHS} sampled HPFC-anchored DA paths x {RUN_STEPS} half-hours')
print(f'  VOM=0 (energy-only upper bound), terminal SoC pinned to {PF_TERMINAL_SOC:.1f} MWh')

pf_values = []
t0 = time.time()
for i, idx in enumerate(ri_idx[:PF_N_PATHS], start=1):
    prices = np.clip(P_da_anchored_mat[idx, :RUN_STEPS], -100.0, 500.0)
    res = solve_perfect_foresight(
        prices, ASSET_VAL, dt_h=DT_H,
        vom_gbp_mwh=0.0,
        terminal_soc_mwh=PF_TERMINAL_SOC,
    )
    pf_values.append(res.objective_gbp)
    if i == 1 or i % max(1, PF_N_PATHS // 5) == 0:
        print(f'  perfect foresight path {i}/{PF_N_PATHS}: GBP{res.objective_gbp:,.0f}')

pf_values = np.asarray(pf_values, dtype=float)
print(f'Perfect foresight complete in {time.time() - t0:.1f}s')

pf_ann = pf_values * annualization_factor
perfect_row = {
    'method': 'Perfect foresight (DA energy)',
    'value_gbp_horizon_mean': float(np.mean(pf_values)),
    'value_gbp_annualized_mean': float(np.mean(pf_ann)),
    'gbp_per_mw_year': float(np.mean(pf_ann)) / ASSET_VAL['power_mw'],
    'n_paths': PF_N_PATHS,
    'window_hh': RUN_STEPS,
    'gate_hh': RUN_STEPS,
    'pv_paths': pf_ann,
    'notes': (
        f'Full-horizon LP on {PF_N_PATHS} sampled HPFC-anchored DA paths; '
        f'energy-only (VOM=0), terminal SoC={PF_TERMINAL_SOC:.0f} MWh; '
        f'upper bound on DA energy arbitrage'
    ),
}
perfect_row


## 6  Summary


In [ ]:
comparison_rows = [
    initial_hourly_row,
    da_rolling_row,
    wd_rolling_row,
    forward_row,
    perfect_row,
]
comparison = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'pv_paths'}
    for r in comparison_rows
])
comparison.insert(0, 'duration_h', VALUATION_DURATION_H)
comparison.insert(1, 'run_mode', PHASE4_RUN_MODE_FOR_SWEEP)
comparison['value_gbp_annualized_m'] = comparison['value_gbp_annualized_mean'] / 1e6
comparison['gbp_per_mw_year_k'] = comparison['gbp_per_mw_year'] / 1e3

# P5 / P95 confidence intervals from per-path distributions
for row_d in comparison_rows:
    pvp = np.asarray(row_d.get('pv_paths', []))
    method = row_d['method']
    mask = comparison['method'] == method
    if len(pvp) >= 2:
        comparison.loc[mask, 'p5_ann_m']  = float(np.percentile(pvp, 5))  / 1e6
        comparison.loc[mask, 'p95_ann_m'] = float(np.percentile(pvp, 95)) / 1e6
    else:
        comparison.loc[mask, 'p5_ann_m']  = comparison.loc[mask, 'value_gbp_annualized_m'].values[0]
        comparison.loc[mask, 'p95_ann_m'] = comparison.loc[mask, 'value_gbp_annualized_m'].values[0]

out_csv = PROCESSED / f'phase4_method_comparison_{DURATION_LABEL}.csv'
comparison.to_csv(out_csv, index=False)

print(f'Saved: {out_csv}')
print(comparison[[
    'method', 'value_gbp_annualized_m', 'p5_ann_m', 'p95_ann_m',
    'gbp_per_mw_year_k', 'n_paths', 'notes'
]].round(3).to_string(index=False))

# ── Two-panel chart: energy-only methods vs full-stack LSMC ──────────────────
ENERGY_ONLY = ['Initial hourly intrinsic', 'DA rolling intrinsic', 'WD rolling intrinsic',
               'Perfect foresight (DA energy)']
FULL_STACK  = ['Forward simulation (LSMC)']
PANEL_COLOUR = {'energy-only': '#4e79a7', 'full-stack': '#f28e2b'}

df_e = comparison[comparison['method'].isin(ENERGY_ONLY)].sort_values('value_gbp_annualized_m')
df_f = comparison[comparison['method'].isin(FULL_STACK)].sort_values('value_gbp_annualized_m')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=False)
fig.suptitle(
    f'Phase 4 method comparison — {DURATION_LABEL} BESS ({PHASE4_RUN_MODE_FOR_SWEEP})\n'
    f'as-of {AS_OF_DATE.date()}',
    fontsize=11,
)

for ax, df, colour, title, panel_label in [
    (axes[0], df_e, PANEL_COLOUR['energy-only'],
     'Panel A — Energy-only DA arbitrage\n(VOM=0, no ancillary)', 'A'),
    (axes[1], df_f, PANEL_COLOUR['full-stack'],
     'Panel B — Full-stack LSMC\n(DA + ancillary, VOM + degradation cost)', 'B'),
]:
    means = df['value_gbp_annualized_m'].values
    xerr_lo = means - df['p5_ann_m'].values
    xerr_hi = df['p95_ann_m'].values - means
    ax.barh(df['method'], means, color=colour, alpha=0.80, zorder=2)
    ax.errorbar(
        means, df['method'],
        xerr=[np.maximum(xerr_lo, 0), np.maximum(xerr_hi, 0)],
        fmt='none', color='#333333', capsize=4, linewidth=1.2, zorder=3,
    )
    ax.set_xlabel('GBP million/year (annualised)')
    ax.set_title(title, fontsize=9)
    ax.grid(axis='x', alpha=0.35, zorder=0)
    for i, (v, n) in enumerate(zip(means, df['n_paths'].values)):
        ax.text(v + max(0.01, xerr_hi[i]) * 1.05, i,
                f'  {v:.2f}m  (n={n})', va='center', fontsize=8)

fig.tight_layout()
out_png = PROCESSED / f'phase4_method_comparison_{DURATION_LABEL}.png'
fig.savefig(out_png, dpi=140, bbox_inches='tight')
print(f'Saved: {out_png}')
plt.show()
